# Pipeline final estilo talleres

Notebook autocontenido para la entrega: carga datos raw, preprocesa con `P04`, entrena el ensemble CV de XGBoost y genera el CSV final de submission.

No usa Comet y no ejecuta submit a Kaggle. Solo guarda el CSV listo para subir.

In [1]:
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import RobustScaler
from xgboost import XGBClassifier

In [2]:
SEED = 42
N_SPLITS = 5
VALID_SIZE = 0.1
MISSING_MARKER = -999.0
EPS = 1e-6
TOP_SIGNAL_PERCENT = 15.75
EXPECTED_TEST_ROWS = 550_000
EXPECTED_SIGNAL_COUNT = 86_625

SPECIAL_COLUMNS = ["EventId", "Weight", "Label", "KaggleSet", "KaggleWeight"]
JET_NUM_COLUMN = "PRI_jet_num"
JET_NUM_VALUES = [0, 1, 2, 3]
FIXED_JET_ONE_HOT_COLUMNS = [f"PRI_jet_num_{value}" for value in JET_NUM_VALUES]

ALLOWED_FEATURE_COLUMNS = [
    "DER_mass_MMC",
    "DER_mass_transverse_met_lep",
    "DER_mass_vis",
    "DER_pt_h",
    "DER_deltaeta_jet_jet",
    "DER_mass_jet_jet",
    "DER_prodeta_jet_jet",
    "DER_deltar_tau_lep",
    "DER_pt_tot",
    "DER_sum_pt",
    "DER_pt_ratio_lep_tau",
    "DER_met_phi_centrality",
    "DER_lep_eta_centrality",
    "PRI_tau_pt",
    "PRI_tau_eta",
    "PRI_tau_phi",
    "PRI_lep_pt",
    "PRI_lep_eta",
    "PRI_lep_phi",
    "PRI_met",
    "PRI_met_phi",
    "PRI_met_sumet",
    "PRI_jet_num",
    "PRI_jet_leading_pt",
    "PRI_jet_leading_eta",
    "PRI_jet_leading_phi",
    "PRI_jet_subleading_pt",
    "PRI_jet_subleading_eta",
    "PRI_jet_subleading_phi",
    "PRI_jet_all_pt",
]

BASE_LOG_COLUMNS = [
    "DER_mass_MMC",
    "DER_mass_transverse_met_lep",
    "DER_mass_vis",
    "DER_pt_h",
    "DER_deltaeta_jet_jet",
    "DER_mass_jet_jet",
    "DER_deltar_tau_lep",
    "DER_pt_tot",
    "DER_sum_pt",
    "DER_pt_ratio_lep_tau",
    "PRI_tau_pt",
    "PRI_lep_pt",
    "PRI_met",
    "PRI_met_sumet",
    "PRI_jet_leading_pt",
    "PRI_jet_subleading_pt",
    "PRI_jet_all_pt",
]
P04_ENGINEERED_LOG_COLUMNS = ["FE_tau_lep_pt_sum"]

FINAL_XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": SEED,
    "tree_method": "hist",
    "n_jobs": -1,
    "missing": np.nan,
    "learning_rate": 0.03,
    "n_estimators": 800,
    "max_depth": 6,
    "min_child_weight": 1,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "scale_pos_weight": 1.5346,
    "gamma": 0.0,
    "max_delta_step": 0,
    "reg_alpha": 0.5,
    "reg_lambda": 2.0,
}

REFERENCE_KAGGLE = {
    "submission": "higgs_submission_p04_xgboost_sw04_cv_ensemble_top15p75pct_20260503_002739.csv",
    "public_score": 3.66729,
    "private_score": 3.65941,
}

In [3]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "roadmap.md").exists() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("No se pudo localizar la raiz del proyecto.")


def abs_delta_phi(phi_a: pd.Series, phi_b: pd.Series) -> pd.Series:
    return ((phi_a - phi_b + np.pi) % (2 * np.pi) - np.pi).abs()


def add_p04_features(X: pd.DataFrame) -> pd.DataFrame:
    result = X.copy()
    result["FE_abs_dphi_tau_lep"] = abs_delta_phi(result["PRI_tau_phi"], result["PRI_lep_phi"])
    result["FE_abs_dphi_tau_met"] = abs_delta_phi(result["PRI_tau_phi"], result["PRI_met_phi"])
    result["FE_abs_dphi_lep_met"] = abs_delta_phi(result["PRI_lep_phi"], result["PRI_met_phi"])
    result["FE_abs_deta_tau_lep"] = (result["PRI_tau_eta"] - result["PRI_lep_eta"]).abs()
    result["FE_tau_lep_pt_sum"] = result["PRI_tau_pt"] + result["PRI_lep_pt"]
    result["FE_tau_pt_fraction"] = result["PRI_tau_pt"] / (result["PRI_tau_pt"] + result["PRI_lep_pt"] + EPS)
    result["FE_met_over_tau_lep_pt"] = result["PRI_met"] / (result["PRI_tau_pt"] + result["PRI_lep_pt"] + EPS)
    result["FE_met_over_sumet"] = result["PRI_met"] / (result["PRI_met_sumet"] + EPS)
    result["FE_pt_balance"] = result["DER_pt_tot"] / (result["DER_sum_pt"] + EPS)
    result["FE_mmc_over_visible_mass"] = result["DER_mass_MMC"] / (result["DER_mass_vis"] + EPS)
    result["FE_transverse_mass_over_visible_mass"] = result["DER_mass_transverse_met_lep"] / (result["DER_mass_vis"] + EPS)
    result["FE_jet_all_pt_fraction"] = result["PRI_jet_all_pt"] / (result["DER_sum_pt"] + EPS)
    result["FE_leading_jet_pt_fraction"] = result["PRI_jet_leading_pt"] / (result["PRI_jet_all_pt"] + EPS)
    result["FE_subleading_jet_pt_fraction"] = result["PRI_jet_subleading_pt"] / (result["PRI_jet_all_pt"] + EPS)
    result["FE_jet_pt_balance"] = (result["PRI_jet_leading_pt"] - result["PRI_jet_subleading_pt"]) / (
        result["PRI_jet_leading_pt"] + result["PRI_jet_subleading_pt"] + EPS
    )
    result["FE_abs_dphi_leading_subleading_jet"] = abs_delta_phi(
        result["PRI_jet_leading_phi"],
        result["PRI_jet_subleading_phi"],
    )
    result["FE_mass_jet_over_sum_pt"] = result["DER_mass_jet_jet"] / (result["DER_sum_pt"] + EPS)
    return result


def validate_raw_higgs(train_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
    feature_cols = [column for column in train_df.columns if column not in SPECIAL_COLUMNS]
    test_feature_cols = [column for column in test_df.columns if column not in SPECIAL_COLUMNS]
    if feature_cols != ALLOWED_FEATURE_COLUMNS:
        raise ValueError("Las features de training.csv no coinciden con la especificacion esperada.")
    if test_feature_cols != feature_cols:
        raise ValueError("Las features de training.csv y test.csv no coinciden.")
    invalid_labels = sorted(set(train_df["Label"].dropna().unique()) - {"b", "s"})
    if invalid_labels:
        raise ValueError(f"Labels invalidos: {invalid_labels}")

In [4]:
class P04Preprocessor(BaseEstimator, TransformerMixin):
    def fit(self, X: pd.DataFrame, y: pd.Series | None = None) -> "P04Preprocessor":
        prepared = self._prepare_features(X)
        self.missing_indicator_cols_ = [
            column for column in prepared.columns
            if column != JET_NUM_COLUMN and prepared[column].isna().any()
        ]
        continuous = prepared.drop(columns=[JET_NUM_COLUMN])
        self.continuous_cols_ = continuous.columns.tolist()
        self.log_cols_ = [
            column for column in BASE_LOG_COLUMNS + P04_ENGINEERED_LOG_COLUMNS
            if column in self.continuous_cols_
        ]
        logged = self._apply_log1p(continuous)
        self.imputer_ = SimpleImputer(strategy="median")
        imputed = self.imputer_.fit_transform(logged[self.continuous_cols_])
        self.scaler_ = RobustScaler(with_centering=True, with_scaling=True, quantile_range=(25.0, 75.0))
        self.scaler_.fit(imputed)
        self.final_feature_order_ = (
            self.continuous_cols_
            + [f"is_missing_{column}" for column in self.missing_indicator_cols_]
            + FIXED_JET_ONE_HOT_COLUMNS
        )
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        prepared = self._prepare_features(X)
        indicator_frame = pd.DataFrame(index=prepared.index)
        for column in self.missing_indicator_cols_:
            indicator_frame[f"is_missing_{column}"] = prepared[column].isna().astype(np.float32)

        jet_one_hot = pd.DataFrame(
            {
                f"PRI_jet_num_{value}": (prepared[JET_NUM_COLUMN].astype(int) == value).astype(np.float32)
                for value in JET_NUM_VALUES
            },
            index=prepared.index,
        )
        continuous = prepared.drop(columns=[JET_NUM_COLUMN])[self.continuous_cols_]
        logged = self._apply_log1p(continuous)
        imputed = self.imputer_.transform(logged[self.continuous_cols_])
        scaled = self.scaler_.transform(imputed)
        scaled_frame = pd.DataFrame(scaled, columns=self.continuous_cols_, index=prepared.index)
        result = pd.concat([scaled_frame, indicator_frame, jet_one_hot], axis=1)
        return result[self.final_feature_order_].astype(np.float32)

    def _prepare_features(self, X: pd.DataFrame) -> pd.DataFrame:
        if JET_NUM_COLUMN not in X.columns:
            raise ValueError("PRI_jet_num no existe antes del preprocesamiento.")
        return add_p04_features(X)

    def _apply_log1p(self, X: pd.DataFrame) -> pd.DataFrame:
        result = X.copy()
        for column in self.log_cols_:
            values = result[column].dropna()
            if (values < 0).any():
                raise ValueError(f"{column} tiene valores negativos y no admite log1p.")
            result[column] = np.log1p(result[column])
        return result


class CVXGBoostEnsemble(BaseEstimator, ClassifierMixin):
    def __init__(self, params: dict[str, object], n_splits: int = N_SPLITS, seed: int = SEED) -> None:
        self.params = params
        self.n_splits = n_splits
        self.seed = seed

    def fit(self, X: pd.DataFrame, y: pd.Series, sample_weight: pd.Series | None = None) -> "CVXGBoostEnsemble":
        self.models_ = []
        self.fit_seconds_ = []
        cv = StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=self.seed)
        for fold_id, (train_idx, _) in enumerate(cv.split(X, y), start=1):
            model = XGBClassifier(**self.params)
            start = time.perf_counter()
            fit_kwargs = {}
            if sample_weight is not None:
                fit_kwargs["sample_weight"] = sample_weight.iloc[train_idx].reset_index(drop=True)
            model.fit(
                X.iloc[train_idx].reset_index(drop=True),
                y.iloc[train_idx].reset_index(drop=True),
                **fit_kwargs,
            )
            self.fit_seconds_.append(time.perf_counter() - start)
            if model.classes_.tolist() != [0, 1]:
                raise ValueError(f"Orden inesperado de clases en fold {fold_id}: {model.classes_.tolist()}")
            self.models_.append(model)
        return self

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        fold_scores = [model.predict_proba(X)[:, 1] for model in self.models_]
        signal_scores = np.column_stack(fold_scores).mean(axis=1)
        return np.column_stack([1.0 - signal_scores, signal_scores])

    def predict_fold_scores(self, X: pd.DataFrame) -> pd.DataFrame:
        return pd.DataFrame(
            {f"score_fold_{idx}": model.predict_proba(X)[:, 1] for idx, model in enumerate(self.models_, start=1)}
        )

In [5]:
class FinalHiggsPublicPipeline:
    def __init__(self) -> None:
        self.preprocessor = P04Preprocessor()
        self.model = CVXGBoostEnsemble(params=FINAL_XGB_PARAMS, n_splits=N_SPLITS, seed=SEED)

    def fit(self, train_df: pd.DataFrame) -> "FinalHiggsPublicPipeline":
        X_full = train_df[ALLOWED_FEATURE_COLUMNS].replace(MISSING_MARKER, np.nan)
        y_full = train_df["Label"].map({"b": 0, "s": 1}).astype(np.int8)
        weight_full = train_df["Weight"].astype(np.float64)
        train_idx, valid_idx = train_test_split(
            X_full.index.to_numpy(),
            test_size=VALID_SIZE,
            random_state=SEED,
            stratify=y_full,
        )

        X_train_raw = X_full.loc[train_idx].reset_index(drop=True)
        X_valid_raw = X_full.loc[valid_idx].reset_index(drop=True)
        y_train = y_full.loc[train_idx].reset_index(drop=True)
        y_valid = y_full.loc[valid_idx].reset_index(drop=True)
        w_train = weight_full.loc[train_idx].reset_index(drop=True)
        w_valid = weight_full.loc[valid_idx].reset_index(drop=True)

        self.preprocessor.fit(X_train_raw)
        X_train_processed = self.preprocessor.transform(X_train_raw)
        X_valid_processed = self.preprocessor.transform(X_valid_raw)
        X_model = pd.concat([X_train_processed, X_valid_processed], ignore_index=True)
        y_model = pd.concat([y_train, y_valid], ignore_index=True)
        w_model = pd.concat([w_train, w_valid], ignore_index=True)

        self.model.fit(X_model, y_model, sample_weight=w_model)
        self.metadata_ = {
            "X_train_processed_shape": list(X_train_processed.shape),
            "X_valid_processed_shape": list(X_valid_processed.shape),
            "X_model_shape": list(X_model.shape),
            "fit_seconds_per_fold": self.model.fit_seconds_,
            "fit_seconds_total": float(sum(self.model.fit_seconds_)),
        }
        return self

    def predict_scores(self, test_df: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
        X_test_raw = test_df[ALLOWED_FEATURE_COLUMNS].replace(MISSING_MARKER, np.nan).reset_index(drop=True)
        X_test_processed = self.preprocessor.transform(X_test_raw)
        fold_scores = self.model.predict_fold_scores(X_test_processed)
        mean_scores = fold_scores.mean(axis=1).to_numpy(dtype=np.float64)
        score_frame = pd.concat(
            [test_df[["EventId"]].reset_index(drop=True).astype({"EventId": np.int64}), fold_scores],
            axis=1,
        )
        score_frame["score_mean"] = mean_scores
        self.metadata_["X_test_processed_shape"] = list(X_test_processed.shape)
        return score_frame, mean_scores


def top_percent_classes(scores: np.ndarray, percent: float) -> np.ndarray:
    n_signal = int(np.ceil(len(scores) * percent / 100.0))
    order = np.argsort(scores)[::-1]
    classes = np.full(len(scores), "b", dtype=object)
    classes[order[:n_signal]] = "s"
    return classes


def build_submission(event_id: pd.Series, scores: np.ndarray) -> pd.DataFrame:
    submission = pd.DataFrame(
        {
            "EventId": event_id.to_numpy(dtype=np.int64),
            "RankOrder": pd.Series(scores).rank(method="first", ascending=True).astype(np.int64),
            "Class": top_percent_classes(scores, TOP_SIGNAL_PERCENT),
        }
    )[["EventId", "RankOrder", "Class"]]
    validate_submission(submission, event_id)
    return submission


def validate_submission(submission: pd.DataFrame, expected_event_id: pd.Series) -> None:
    if list(submission.columns) != ["EventId", "RankOrder", "Class"]:
        raise ValueError("Columnas invalidas para submission.")
    if len(submission) != EXPECTED_TEST_ROWS:
        raise ValueError(f"Cantidad de filas incorrecta: {len(submission)}")
    if not submission["EventId"].reset_index(drop=True).equals(expected_event_id.reset_index(drop=True)):
        raise ValueError("EventId no coincide con test.csv.")
    if submission["RankOrder"].nunique() != len(submission):
        raise ValueError("RankOrder no es unico.")
    ranks = submission["RankOrder"].to_numpy(dtype=np.int64)
    if not np.array_equal(np.sort(ranks), np.arange(1, len(submission) + 1)):
        raise ValueError("RankOrder debe cubrir 1..n.")
    classes = set(submission["Class"].unique())
    if not classes <= {"s", "b"}:
        raise ValueError(f"Clases invalidas: {classes}")
    signal_count = int((submission["Class"] == "s").sum())
    if signal_count != EXPECTED_SIGNAL_COUNT:
        raise ValueError(f"Signal count inesperado: {signal_count} != {EXPECTED_SIGNAL_COUNT}")

In [6]:
project_root = find_project_root()
data_dir = project_root / "data" / "higgs-boson"
results_dir = project_root / "100. Pipeline" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

train_path = data_dir / "training.csv"
test_path = data_dir / "test.csv"
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
validate_raw_higgs(train_df, test_df)

summary = {
    "train_path": str(train_path),
    "test_path": str(test_path),
    "train_shape": train_df.shape,
    "test_shape": test_df.shape,
    "pipeline": "P04",
    "model": "CV ensemble XGBoost SW04",
    "threshold_policy": f"top {TOP_SIGNAL_PERCENT}% por ranking de score",
    "reference_kaggle": REFERENCE_KAGGLE,
}
summary

{'train_path': '/home/santig14/fing/taa/1_TallerAA/data/higgs-boson/training.csv',
 'test_path': '/home/santig14/fing/taa/1_TallerAA/data/higgs-boson/test.csv',
 'train_shape': (250000, 33),
 'test_shape': (550000, 31),
 'pipeline': 'P04',
 'model': 'CV ensemble XGBoost SW04',
 'threshold_policy': 'top 15.75% por ranking de score',
 'reference_kaggle': {'submission': 'higgs_submission_p04_xgboost_sw04_cv_ensemble_top15p75pct_20260503_002739.csv',
  'public_score': 3.66729,
  'private_score': 3.65941}}

In [7]:
pipeline = FinalHiggsPublicPipeline()
pipeline.fit(train_df)
scores_df, scores = pipeline.predict_scores(test_df)
submission_df = build_submission(test_df["EventId"].astype(np.int64), scores)

scores_path = results_dir / "scores_pipeline_estilo_talleres_p04_xgb_cv_ensemble_top15p75.csv"
submission_path = results_dir / "higgs_submission_pipeline_estilo_talleres_p04_xgb_cv_ensemble_top15p75.csv"
metadata_path = results_dir / "metadata_pipeline_estilo_talleres_p04_xgb_cv_ensemble_top15p75.json"

scores_df.to_csv(scores_path, index=False)
submission_df.to_csv(submission_path, index=False)

metadata = {
    **summary,
    "hyperparameters": FINAL_XGB_PARAMS,
    "validation": {
        "rows": int(len(submission_df)),
        "signal_count": int((submission_df["Class"] == "s").sum()),
        "background_count": int((submission_df["Class"] == "b").sum()),
        "signal_percent": float((submission_df["Class"] == "s").mean() * 100.0),
        "rank_order_unique": int(submission_df["RankOrder"].nunique()),
        "event_id_unique": int(submission_df["EventId"].nunique()),
    },
    "pipeline_metadata": pipeline.metadata_,
    "scores_path": str(scores_path),
    "submission_path": str(submission_path),
}
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False))

print(f"scores_path: {scores_path}")
print(f"submission_path: {submission_path}")
print(f"metadata_path: {metadata_path}")
metadata["validation"]

scores_path: /home/santig14/fing/taa/1_TallerAA/100. Pipeline/results/scores_pipeline_estilo_talleres_p04_xgb_cv_ensemble_top15p75.csv
submission_path: /home/santig14/fing/taa/1_TallerAA/100. Pipeline/results/higgs_submission_pipeline_estilo_talleres_p04_xgb_cv_ensemble_top15p75.csv
metadata_path: /home/santig14/fing/taa/1_TallerAA/100. Pipeline/results/metadata_pipeline_estilo_talleres_p04_xgb_cv_ensemble_top15p75.json


{'rows': 550000,
 'signal_count': 86625,
 'background_count': 463375,
 'signal_percent': 15.75,
 'rank_order_unique': 550000,
 'event_id_unique': 550000}

In [8]:
display(scores_df.head())
display(submission_df.head())
submission_df["Class"].value_counts(normalize=True).rename("rate") * 100

,EventId,score_fold_1,score_fold_2,score_fold_3,score_fold_4,score_fold_5,score_mean
0,350000,0.000018,0.000014,0.000016,0.000014,0.000016,0.000016
1,350001,0.000328,0.000292,0.000312,0.000285,0.000296,0.000302
2,350002,0.008626,0.007081,0.005109,0.007439,0.006874,0.007026
3,350003,0.098081,0.128280,0.091811,0.118606,0.144587,0.116273
4,350004,0.000442,0.000315,0.000365,0.000383,0.000369,0.000375


,EventId,RankOrder,Class
0,350000,6113,b
1,350001,155954,b
2,350002,356058,b
3,350003,495915,s
4,350004,167317,b


Class
b    84.25
s    15.75
Name: rate, dtype: float64